# Evidence snapshot to Drive (YL-004B, stage 02)

For every company you marked `READY_FOR_TEASER` in the database: takes its sources from the database, makes a dated text snapshot of each page (no AI) and saves **one file `<company_id>.md` per company** in the Drive folder **YL004B evidence**.
The Make Teaser Generator reads this file from that folder. Nothing is copied by hand.

**Before the first run (once):**
1. Colab Secrets (key icon on the left): add `SUPABASE_DB_URL` = the connection string of the Supabase project "Yl004B prototype" (Connect → Session pooler, with the password). Switch on "Notebook access". Never paste it into a cell.
2. The Google account you sign in with in Step 2 must be able to edit the folder YL004B evidence.

Run the cells from top to bottom.

### Step 1. Get the code

In [ ]:
import os

REPO_URL = "https://github.com/Yul04ek/ai-content-claim-check.git"
BRANCH = "main"
FOLDER = "/content/claim-check"

if not os.path.exists(FOLDER):
    !git clone -q -b {BRANCH} {REPO_URL} {FOLDER}
else:
    !git -C {FOLDER} pull -q origin {BRANCH}
%cd {FOLDER}
!git log --oneline -1
!pip install -q requests beautifulsoup4 lxml psycopg2-binary

### Step 2. Sign in to Google (Drive access)
A window asks which Google account to use. Choose the account that can edit the folder YL004B evidence.

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
drive = build("drive", "v3")

EVIDENCE_FOLDER_ID = "14vBlGO9eX8v5N8MMGNHpEE1BtmK7glVi"   # folder "YL004B evidence"

# Check that the folder is visible to this account
folder = drive.files().get(fileId=EVIDENCE_FOLDER_ID, fields="name, capabilities(canAddChildren)",
                           supportsAllDrives=True).execute()
print("Folder:", folder["name"], "| can write:", folder["capabilities"]["canAddChildren"])

### Step 3. The queue from the database
Takes every company you marked `READY_FOR_TEASER` in the table `companies` (reviewed, main company, no teaser yet) and its sources: the whole group with duplicates, identical addresses once. Nothing to type here.

In [ ]:
import psycopg2
from google.colab import userdata

def db_rows(sql, params=()):
    """Run one read-only query and return the rows as dicts."""
    with psycopg2.connect(userdata.get("SUPABASE_DB_URL")) as conn:
        conn.set_session(readonly=True)          # this notebook never changes the database
        with conn.cursor() as cur:
            cur.execute(sql, params)
            cols = [c.name for c in cur.description]
            return [dict(zip(cols, r)) for r in cur.fetchall()]

queue = db_rows("select company_id, label from public.teaser_candidates order by label")
assert queue, "The queue is empty: set teaser_state = READY_FOR_TEASER (and reviewer_ref) for a company in the table companies."

sources = {}   # company_id -> its sources
for c in queue:
    sources[c["company_id"]] = db_rows(
        "select evidence_id, belongs_to, label_in_source, source_url from public.teaser_sources "
        "where company_id = %s order by evidence_id", (c["company_id"],))
    print(f"{c['label']}  ({c['company_id']}): {len(sources[c['company_id']])} sources")
    for s in sources[c["company_id"]]:
        own = "" if s["belongs_to"] == c["company_id"] else f"   [from duplicate {s['label_in_source']}]"
        print(f"   {s['evidence_id']}  {s['source_url']}{own}")

### Step 4. Read the pages
Every address is read once. Pages that cannot be read are listed as SKIPPED with the reason (not https, forbidden by robots.txt, page not found, built by scripts).

In [ ]:
import sys
sys.path.insert(0, "/content/claim-check")
from evidence_fetch.fetch import fetch_snapshot

snaps, skipped = {}, []          # snaps: evidence_id -> snapshot; skipped: sources to upload by hand
for c in queue:
    print(f"\n== {c['label']}")
    for s in sources[c["company_id"]]:
        url = s["source_url"].strip()
        try:
            snap = fetch_snapshot(url)
        except Exception as e:
            skipped.append(s)
            print(f"SKIPPED {s['evidence_id']} {url}\n   reason: {e}")
            continue
        words = len(snap["text"].split())
        if words < 80:           # pages built by scripts give almost no text
            skipped.append(s)
            print(f"SKIPPED {s['evidence_id']} {url}\n   reason: only {words} words, the page is probably built by scripts")
            continue
        snaps[s["evidence_id"]] = snap
        print(f"OK      {s['evidence_id']} {url}: {words} words, page date: {snap['page_date'] or 'none shown'}")
print(f"\n{len(snaps)} read, {len(skipped)} skipped")

### Step 5 (only if pages were skipped). Saved pages instead
For each skipped address: open it in the browser, save it (Ctrl+S, "Webpage, HTML only"), then upload the file when the cell asks.
To leave an address out, press Cancel in the upload window.

In [ ]:
from google.colab import files
from evidence_fetch.fetch import snapshot_from_html

for s in skipped:
    print(f"\n{s['evidence_id']}  {s['source_url']}\nUpload the saved page (or Cancel to leave it out):")
    try:
        up = files.upload()
    except Exception:
        up = {}
    if not up:
        print("   left out")
        continue
    html = next(iter(up.values())).decode("utf-8", errors="replace")
    snap = snapshot_from_html(html, s["source_url"].strip())
    snaps[s["evidence_id"]] = snap
    print(f"   OK: {len(snap['text'].split())} words, page date: {snap['page_date'] or 'none shown'}")

### Step 6. Save to Drive
One file `<company_id>.md` per company in YL004B evidence. If the file already exists, it is **replaced** (Make always finds exactly one).
A company with no source read is not saved.

In [ ]:
import io
from googleapiclient.http import MediaIoBaseUpload
from evidence_fetch.fetch import to_source_block

def save_to_drive(name, text):
    """Create or replace one text file in the evidence folder and return its link."""
    media = MediaIoBaseUpload(io.BytesIO(text.encode("utf-8")), mimetype="text/markdown", resumable=False)
    found = drive.files().list(
        q=f"'{EVIDENCE_FOLDER_ID}' in parents and name = '{name}' and trashed = false",
        fields="files(id)", supportsAllDrives=True, includeItemsFromAllDrives=True).execute()["files"]
    if found:
        if len(found) > 1:
            print(f"   WARNING: {len(found)} files named {name}. Make reads only one: remove the extra ones.")
        f = drive.files().update(fileId=found[0]["id"], media_body=media, fields="webViewLink", supportsAllDrives=True).execute()
        return "replaced", f["webViewLink"]
    f = drive.files().create(body={"name": name, "parents": [EVIDENCE_FOLDER_ID]}, media_body=media,
                             fields="webViewLink", supportsAllDrives=True).execute()
    return "saved", f["webViewLink"]

for c in queue:
    order = [s["evidence_id"] for s in sources[c["company_id"]] if s["evidence_id"] in snaps]   # database order
    left_out = [s["evidence_id"] for s in sources[c["company_id"]] if s["evidence_id"] not in snaps]
    print(f"\n== {c['label']}: {len(order)} sources, left out: {left_out or 'none'}")
    if not order:
        print("   NOT saved: no source was read")
        continue
    for e in order:
        print(f"   {e}: " + " / ".join(snaps[e]["text"].splitlines()[:2])[:150])
    block = "\n".join(to_source_block(snaps[e]) for e in order)
    action, link = save_to_drive(f"{c['company_id']}.md", block)
    print(f"   {action}: {c['company_id']}.md  {link}")
print("\nNext: run the Make Teaser Generator.")